# 4. Calibrate the classifier

Calibration gate for **altendor**. This notebook runs the LLM classifier on every row in
`altendor/data/calibration/labeled.jsonl` and scores predictions against the hand-labelled
ground truth.

**Gate**: if **MAE ≤ 8 dB** and **macro-F1 ≥ 0.8**, the gate passes and you can run the
full pipeline. Otherwise iterate on `altendor/classify/prompts.py` and re-run this
notebook before kicking off a real run.

Calibration is a **one-time validation** — it does not depend on any `RUN_ID` and reads
from the curated labels file, not a run-specific input.

See `PIPELINE_PLAN.md` § S21 / S12 for the contract.

## Setup

Load `.env` (for `ANTHROPIC_API_KEY`) and resolve the repo root.

In [1]:
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent.resolve()
load_dotenv(REPO_ROOT / ".env")
print(f"Repo root: {REPO_ROOT}")

Repo root: /home/automathis/Documents/Codebases/ICSSI-2026-Hackaton


## Load labelled calibration rows

In [2]:
from altendor.classify.calibration import load_labeled_jsonl

LABELED_PATH = REPO_ROOT / "altendor" / "data" / "calibration" / "labeled.jsonl"
examples = load_labeled_jsonl(LABELED_PATH)
print(f"Loaded {len(examples)} calibration rows")

Loaded 10 calibration rows


## Run the classifier on each row

Wraps each labelled row into a `ResolvedPost` + `PaperCtx` and calls `classify_post`.
Failures are logged and skipped — the surviving `(example, result)` pairs are scored
below.

In [3]:
import anthropic
from altendor.classify.classifier import classify_post, PaperCtx
from altendor.enrich.text_resolver import ResolvedPost

client = anthropic.Anthropic()
pairs = []
for i, ex in enumerate(examples, start=1):
    post = ResolvedPost(
        post_id=f"calib-{i:03d}",
        platform="other",
        text=ex.post_text,
        author_handle=None,
        author_id=None,
        url="",
        created_at="",
        raw_title=ex.post_text,
        text_confidence="high",
    )
    paper = PaperCtx(title=ex.paper_title, abstract=ex.paper_abstract or None)
    try:
        result = classify_post(client, post, paper)
    except Exception as exc:
        print(f"  row {i}: classifier failed ({exc})")
        continue
    pairs.append((ex, result))
print(f"Classified {len(pairs)} / {len(examples)} rows.")

Classified 10 / 10 rows.


## Score and render the report

`score(pairs)` computes MAE on the dB endorsement strength and macro-F1 on the
endorsement kind. `render_report_markdown` produces the human-readable rollup; we
also persist it to `altendor/data/calibration/last_report.md` for later reference.

In [4]:
from altendor.classify.calibration import score, render_report_markdown
from IPython.display import Markdown

report = score(pairs)
md = render_report_markdown(report)
(REPO_ROOT / "altendor" / "data" / "calibration" / "last_report.md").write_text(md)
Markdown(md)

## Calibration Report
- n_total: 10
- MAE (dB, endorsement subset): 3.60  (threshold 8.0, n=5)
- Macro-F1: 1.000  (threshold 0.8)
- Passes gate: True

### Per-kind F1
| Kind | F1 |
|------|-----|
| endorsement | 1.000 |
| flag | 1.000 |
| irrelevant | 1.000 |

### Confusion Matrix (rows = gold, cols = predicted)
| gold \ pred | endorsement | flag | irrelevant |
|---|---|---|---|
| endorsement | 5 | 0 | 0 |
| flag | 0 | 3 | 0 |
| irrelevant | 0 | 0 | 2 |

## Gate

Raises if calibration fails so a notebook-runner / CI invocation surfaces the failure.
If this cell raises, iterate on `altendor/classify/prompts.py` and re-run from the top
before launching the main pipeline.

In [5]:
if not report.passes:
    raise RuntimeError(
        f"Calibration gate FAILED: MAE={report.mae_dB}, F1={report.kind_f1_macro:.3f}. "
        "Iterate on altendor/classify/prompts.py before running the full pipeline."
    )
print(f"Calibration gate PASSED: MAE={report.mae_dB}, F1={report.kind_f1_macro:.3f}")

Calibration gate PASSED: MAE=3.6, F1=1.000
